# GiziLens – Training Modul 1 di Kaggle

**Sebelum menjalankan:**
1. Upload folder `dataset/` hasil `merge_datasets.py` **beserta `classes.json`** sebagai Kaggle Dataset, lalu *Add Data* ke notebook ini.
2. Settings (panel kanan): Accelerator = **GPU P100** atau **GPU T4 x2** (kode memakai 1 GPU), Internet = **On** (butuh akun terverifikasi nomor HP).
3. Jalankan semua sel. Setelah selesai, pilih **Save Version → Save & Run All** supaya hasil di `/kaggle/working` tersimpan dan bisa diunduh dari tab Output.

In [ ]:
# ====== PENGATURAN ======
MODEL     = "yolo11n-seg.pt"   # nano dulu; naik ke yolo11s-seg.pt kalau hasil kurang
EPOCHS    = 100
IMGSZ     = 640
BATCH     = 16                 # turunkan ke 8 kalau CUDA out of memory
PRIORITAS = 1                  # 1 = kelas inti, 2 = + kelas umum, 3 / None = semua kelas

In [ ]:
import os, glob, json, shutil, subprocess, sys
os.environ["WANDB_DISABLED"] = "true"
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "ultralytics"], check=True)
subprocess.run(["nvidia-smi"])

In [ ]:
import yaml
from pathlib import Path

# --- cari dataset di /kaggle/input (yang punya images/train) ---
cands = [Path(p) for p in glob.glob("/kaggle/input/**/gizilens.yaml", recursive=True)]
cands = [p for p in cands if (p.parent / "images" / "train").exists()]
assert cands, "gizilens.yaml + images/train tidak ditemukan di /kaggle/input. Cek struktur dataset yang diupload."
src = cands[0].parent
print("Dataset sumber:", src)

# --- salin ke /kaggle/working (input read-only; Ultralytics menulis cache label) ---
DST = Path("/kaggle/working/dataset")
if DST.exists():
    shutil.rmtree(DST)
shutil.copytree(src, DST)

# --- tulis ulang yaml: 'path' dari merge berisi path laptop (mis. C:\\Projects\\...) ---
cfg = yaml.safe_load(open(DST / "gizilens.yaml", encoding="utf-8"))
cfg["path"] = str(DST)
for k in ("train", "val", "test"):
    if k in cfg and not (DST / cfg[k]).exists():
        print(f"PERINGATAN: folder '{cfg[k]}' tidak ada, key '{k}' dihapus")
        cfg.pop(k)
DATA_YAML = "/kaggle/working/gizilens.yaml"
yaml.safe_dump(cfg, open(DATA_YAML, "w", encoding="utf-8"), allow_unicode=True, sort_keys=False)

n_img = {s: len(os.listdir(DST / "images" / s)) for s in ("train", "val", "test") if (DST / "images" / s).exists()}
print("Jumlah gambar:", n_img, "| jumlah kelas:", len(cfg["names"]))

In [ ]:
# --- pilih kelas berdasarkan prioritas (butuh classes.json di dataset yang diupload) ---
extra = {}
cj = glob.glob("/kaggle/input/**/classes.json", recursive=True)
if PRIORITAS and PRIORITAS < 3 and cj:
    classes = json.load(open(cj[0], encoding="utf-8"))["classes"]
    ids = [c["id"] for c in classes if c["prioritas"] <= PRIORITAS]
    extra["classes"] = ids
    print(f"Melatih {len(ids)} kelas (prioritas <= {PRIORITAS})")
elif PRIORITAS and PRIORITAS < 3:
    print("classes.json tidak ditemukan -> melatih SEMUA kelas")

In [ ]:
from ultralytics import YOLO

model = YOLO(MODEL)
model.train(
    data=DATA_YAML, epochs=EPOCHS, imgsz=IMGSZ, batch=BATCH,
    device=0, workers=2,
    project="/kaggle/working/runs", name="gizilens", exist_ok=True,
    patience=30, seed=42, save_period=10,
    # foto ompreng dari atas -> flip vertikal & rotasi aman
    flipud=0.5, fliplr=0.5, degrees=15.0, scale=0.3, translate=0.1,
    hsv_h=0.015, hsv_s=0.5, hsv_v=0.4,
    mosaic=1.0, close_mosaic=10,
    # ompreng & sekat membungkus makanan: jangan biarkan mask-nya berlubang
    overlap_mask=False,
    **extra,
)

In [ ]:
# --- evaluasi: mAP per kelas ---
BEST = "/kaggle/working/runs/gizilens/weights/best.pt"
m = YOLO(BEST)
split = "test" if "test" in cfg else "val"
r = m.val(data=DATA_YAML, imgsz=IMGSZ, split=split, device=0)
print(f"Split: {split}")
print(f"Box  mAP50: {r.box.map50:.3f} | mAP50-95: {r.box.map:.3f}")
print(f"Mask mAP50: {r.seg.map50:.3f} | mAP50-95: {r.seg.map:.3f}")
print("\nmAP50-95 mask per kelas:")
for i in r.seg.ap_class_index:
    print(f"  {m.names[int(i)]:24s} {r.seg.maps[int(i)]:.3f}")

In [ ]:
# --- rapikan output: simpan best.pt + hasil di /kaggle/working ---
shutil.copy(BEST, "/kaggle/working/best.pt")
shutil.rmtree("/kaggle/working/dataset", ignore_errors=True)   # jangan ikut tersimpan di Output
print(os.listdir("/kaggle/working"))
# Unduh best.pt dan folder runs/gizilens/ (results.png, confusion_matrix.png) dari tab Output.

In [ ]:
# (Opsional) lihat grafik training
from IPython.display import Image, display
for f in ("results.png", "confusion_matrix_normalized.png"):
    p = f"/kaggle/working/runs/gizilens/{f}"
    if os.path.exists(p):
        display(Image(p))